In [1]:
#!/usr/bin/python3

import os, re
from collections import defaultdict

from UD_dataclasses import *
from mapping import *

import requests
import tarfile, os
from pathlib import Path

### Load UD from source url

In [ ]:

url = "https://lindat.mff.cuni.cz/repository/xmlui/bitstream/handle/11234/1-5901/ud-treebanks-v2.16.tgz"

local_filename = "ud-treebanks-v2.16.tgz"

response = requests.get(url, stream=True)
response.raise_for_status()
with open(local_filename, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print(f"Downloaded to {local_filename}")


Downloaded to ud-treebanks-v2.16.tgz


In [3]:
output_dir = "."
output_dir = Path(output_dir)

print("Extracting ...")
os.makedirs(output_dir, exist_ok=True)

with tarfile.open(local_filename, "r:gz") as tar:
    tar.extractall(output_dir)
    os.remove(local_filename)

print("Extraction completed ...")

Extracting ...
Extraction completed ...


In [42]:


dir_name = local_filename.split('.tgz')[0]
treebanks_in_mapping = ["UD_"+language+"-"+treebank_name for language in map_lang.keys() for treebank_name in map_lang[language].keys()]
all_treebanks = [tb_dir for tb_dir in os.listdir(dir_name)]

print(f"{'\n'+'-'*100+'\n'}Total treebanks in mapping: {len(treebanks_in_mapping)}")

print(f"Treebanks from genre mapping that are not in the {local_filename}: {set(treebanks_in_mapping)-\
    set(all_treebanks)}{'\n'+'-'*100+'\n'}")



----------------------------------------------------------------------------------------------------
Total treebanks in mapping: 61
Treebanks from genre mapping that are not in the ud-treebanks-v2.16.tgz: {'UD_Norwegian-NynorskLIA', 'UD_English-Tweebank'}
----------------------------------------------------------------------------------------------------



Information about the sources of 'UD_English-Tweebank', 'UD_Norwegian-NynorskLIA' is available in `mapping.py`

### Remove Treebanks not in Genre Mapping

In [ ]:
import glob
import shutil

UD_path=os.path.join(output_dir.resolve(), dir_name)

# Remove treebanks not in mapping from the UD folder
for treebank_path in glob.glob(UD_path+'/*'):
    if not os.path.basename(treebank_path) in treebanks_in_mapping:
        shutil.rmtree(treebank_path)


### Load Treebanks with Available Genre Mapping

In [2]:
UD_path = "ud-treebanks-v2.16"
ud = UniversalDependencies.from_directory(UD_path, verbose=False)

### Validate Genre-Specific Patterns for Treebanks in `mapping.py`

- if genre-specific patterns are available for a given treebank, checks if all of them can be found
among the current treebank's patterns

In [4]:
mapping_valid, pattern_dict = ud.validate_patterns_by_treebank()

In [5]:
mapping_valid

True

In [6]:
pattern_dict

defaultdict(dict,
            {'Armenian+ArmTDP': {'mapping_patterns': ['sent_id = nonfiction-005F',
               'sent_id = nonfiction-006Q',
               'sent_id = blog',
               'sent_id = nonfiction-006U',
               'sent_id = fiction',
               'sent_id = news',
               'sent_id = legal'],
              'found_patterns': ['sent_id = nonfiction-005F',
               'sent_id = nonfiction-006Q',
               'sent_id = blog',
               'sent_id = nonfiction-006U',
               'sent_id = fiction',
               'sent_id = news',
               'sent_id = legal']},
             'Armenian+BSUT': {'mapping_patterns': ['sent_id = nonfiction',
               'sent_id = blog',
               'sent_id = fiction',
               'sent_id = news',
               'sent_id = legal',
               'sent_id = government',
               'sent_id = wiki'],
              'found_patterns': ['sent_id = nonfiction',
               'sent_id = blog',
           

### Cluster patterns in treebank and show longest common substrings in clusters

In [ ]:
a_treebank = ud.get_treebanks()[171]
clusters_dict, longest_common_substrings = a_treebank.get_pattern_clusters(verbose=True, min_cluster_size=50)




----------------------------------------------------------------------------------------------------
*Pattern clusters*
Identifier: newdoc id
Treebank: ArmTDP
Language: Western Armenian

----------------------------------------------------------------------------------------------------
Noise: ['blog-003A', 'wiki-002O', 'reviews-002B', 'news-0017', 'news-0015', 'news-0010', 'fiction-000U', 'fiction-000S', 'fiction-000B', 'nonfiction-0009']

----------------------------------------------------------------------------------------------------
*Pattern clusters*
Identifier: sent_id
Treebank: ArmTDP
Language: Western Armenian

----------------------------------------------------------------------------------------------------
Cluster 0: ['blog-003A-000006ST', 'blog-003A-000106SU', 'blog-003A-000206SV', 'blog-003A-000206T0', 'blog-003A-000206T1', 'blog-003A-000206T2', 'blog-003A-000306T3', 'blog-003A-000406T4', 'blog-003A-000406T5', 'blog-003A-000506T6', 'blog-003A-000506T7', 'blog-003A-000

### Build training and development datasets with genre annotation

In [14]:
UD_folder = "ud-treebanks-v2.16"
! python3 build.py {UD_folder}

Prepares for loading UD data ...
Nynorsk data loaded successfully.
Loaded <UniversalDependenciesTreebank (af_afribooms-ud-dev.conllu): 194 sentences>.
Loaded <UniversalDependenciesTreebank (af_afribooms-ud-test.conllu): 425 sentences>.
Loaded <UniversalDependenciesTreebank (af_afribooms-ud-train.conllu): 1315 sentences>.
Loaded <UniversalDependenciesTreebank (hy_armtdp-ud-dev.conllu): 249 sentences>.
Loaded <UniversalDependenciesTreebank (hy_armtdp-ud-test.conllu): 277 sentences>.
Loaded <UniversalDependenciesTreebank (hy_armtdp-ud-train.conllu): 1974 sentences>.
Loaded <UniversalDependenciesTreebank (hy_bsut-ud-dev.conllu): 479 sentences>.
Loaded <UniversalDependenciesTreebank (hy_bsut-ud-test.conllu): 595 sentences>.
Loaded <UniversalDependenciesTreebank (hy_bsut-ud-train.conllu): 1226 sentences>.
Loaded <UniversalDependenciesTreebank (be_hse-ud-dev.conllu): 1301 sentences>.
Loaded <UniversalDependenciesTreebank (be_hse-ud-test.conllu): 1077 sentences>.
Loaded <UniversalDependenciesT

### Compare with the previous version of UD-MULTIGENRE folder 

In [17]:
import filecmp
import os

def compare_folders_recursively(dir1, dir2):
    """
    Recursively compares two directories and reports differences.
    """
    dcmp = filecmp.dircmp(dir1, dir2)

    if dcmp.left_only:
        print(f"Files and folders only in {dir1}: {dcmp.left_only}")
    if dcmp.right_only:
        print(f"Files and folders only in {dir2}: {dcmp.right_only}")
    if dcmp.diff_files:
        print(f"Files with different content: {dcmp.diff_files}")
    if dcmp.funny_files:
        print(f"Files that could not be compared due to errors: {dcmp.funny_files}")

    # Recursively compare subdirectories
    for common_dir in dcmp.common_dirs:
        path1 = os.path.join(dir1, common_dir)
        path2 = os.path.join(dir2, common_dir)
        print(f"\nComparing common subdirectory: {path1} and {path2}")
        compare_folders_recursively(path1, path2)

folder_A = '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev'
folder_B = '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre'

print(f"Starting comparison between '{folder_A}' and '{folder_B}'")
compare_folders_recursively(folder_A, folder_B)

Starting comparison between '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev' and '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre'
Files and folders only in /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev: ['.DS_Store']

Comparing common subdirectory: /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/QA and /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/QA
Files and folders only in /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/QA: ['.DS_Store']

Comparing common subdirectory: /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/QA/UD_Dutch-Alpino and /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/QA/UD_Dutch-Alpino
Files and folders only in /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/QA/UD_Dutch-Alpino: ['LICENSE.txt']
Files with different content: ['QA-dev.conllu', 'QA-dev.txt', 'QA-train.conllu', 'QA-train.txt']

Com

In [18]:
import os
import shutil
import re

def copy_license_files(source_dir, destination_dir):
    """
    Copies 'LICENSE.txt' files 

    Args:
        source_dir (str): The path to the source directory 
        destination_dir (str): The path to the destination directory
    """
    
    for folder_name in sorted(os.listdir(source_dir)):
        
        source_path = os.path.join(source_dir, folder_name)
        
        print(f"Processing folder: {source_path}")

        for root, dirs, files in os.walk(source_path):
            if 'LICENSE.txt' in files:
                source_file_path = os.path.join(root, 'LICENSE.txt')
                try:
                    relative_path = os.path.relpath(source_file_path, source_dir)
                    destination_file_path = os.path.join(destination_dir, relative_path)

                    shutil.copy2(source_file_path, destination_file_path)
                    print(f"  -> Copied '{os.path.basename(source_file_path)}' to '{destination_file_path}'")
                
                except FileNotFoundError:
                    print(f"  -> Skipping. File 'LICENSE.txt' not found in '{root}'")
                
                except Exception as e:
                    print(f"  -> An unexpected error occurred: {e}")

source_folder = '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev'
destination_folder = '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre'

copy_license_files(source_folder, destination_folder)

Processing folder: /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/.DS_Store
Processing folder: /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/QA
  -> Copied 'LICENSE.txt' to '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/QA/UD_English-EWT/LICENSE.txt'
  -> Copied 'LICENSE.txt' to '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/QA/UD_Russian-Taiga/LICENSE.txt'
  -> Copied 'LICENSE.txt' to '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/QA/UD_Dutch-Alpino/LICENSE.txt'
  -> Copied 'LICENSE.txt' to '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/QA/UD_Italian-ISDT/LICENSE.txt'
Processing folder: /zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/temp/train:dev/academic
  -> Copied 'LICENSE.txt' to '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/build/UD-multigenre/academic/UD_Polish-LFG/LICENSE.txt'
  -> Copied 'LICENSE.txt' to '/zpool/aurora-main/scratch/vera/UD-MULTIGENRE/